# Frozen-trunk detection head — Stage 0c on Brackish (Colab, cloud-native)

**Question.** Does a ~3.4M-parameter CenterNet-style head on *frozen* DINOv3 ConvNeXt-B features detect fish competitively on the Brackish source of the Community Fish Detection Dataset (CFD), scored by the same `pycocotools` harness as the released RF-DETR baselines on the *identical* val images?

**Layout — nothing touches the laptop.**

| Location | Holds |
|---|---|
| `/content/` (VM SSD, ephemeral) | CFD metadata, the Brackish images (re-fetched each session — cheap on the LILA/GCS pipe) |
| Google Drive `frozen-trunk-detection/` | converted DINOv3 backbone weights (`weights/`), run dirs (`runs/<name>/` — `best.pt`, `history.json`, `curves.png`, `predictions.json`), `results/` (manifest, baseline predictions, `results.csv`, viz) |

**Do not put images on Drive** — per-file Drive API latency makes 12k JPEG reads crawl.

**Runtime.** T4/L4 is enough here: only the head trains and the backbone runs under `no_grad`, so the run is very likely CPU-bound on JPEG decode + augmentation. Cell 8 measures GPU utilisation before you spend on anything bigger. If the VM dies, re-run cells 1–7 (minutes) — every artefact of value is already on Drive.

**Rules written before the numbers** (also in `output/frozen-trunk-detection/report.md`): never compare against the README's `.609` (different split — we re-run their weights instead); the baselines likely *saw* these val images, so their number is biased in their favour — our win is conservative, our loss is ambiguous and is reported as ambiguous; report trainable params **and** total inference FLOPs.

## 1 · Drive + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

## 2 · Code — clone the `poc/detection-head` branch (public repo) and install

In [ ]:
%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3 · Machine

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, torch
print('vCPUs:', os.cpu_count(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
from cropcounter.dinov3_pyramid import amp_dtype
_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('compute capability:', torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
      '| AMP dtype the package will use:', amp_dtype(_dev), '(T4 = sm_75 -> float16 + GradScaler; Ampere+ -> bfloat16)')
!df -h /content | tail -1

## 4 · Backbone weights + metric-reproduction check

The DINOv3 checkpoint is Meta-gated; the file on Drive is the ungated timm re-host converted to Meta's parameter names (see vault memory `reference_dinov3_gated_weights`). A strict load only proves the *shapes* match, so we verify by **reproducing a metric**: the shipped wheat decoder `weights/decoder_best.pt` (tracked in git) on the six committed example val images must reproduce the committed counts in `notebooks/runs/inference/examples_val/counts.csv` (±1 from CUDA bf16 autocast is acceptable).

In [ ]:
import shutil, os, csv, torch
from pathlib import Path
os.makedirs(f'{REPO}/weights', exist_ok=True)
src = f'{DRIVE}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
dst = f'{REPO}/weights/dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.exists(dst):
    shutil.copy(src, dst)   # one-off 350 MB Drive read; local SSD from here on
print(os.path.getsize(dst) / 1e6, 'MB')

from cropcounter.train import load_checkpoint
from cropcounter.crop_dataset import CropTileDataset, load_records
from cropcounter.inference import predict_prob, decode_in_bounds
from cropcounter.metrics import evaluate
from torch.utils.data import DataLoader
from cropcounter.crop_dataset import collate_val

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_checkpoint(Path('weights/decoder_best.pt'), device, weights_dir=Path('weights'))
model.eval()
print('point-task decoder loaded strictly; task =', cfg.task)

recs = load_records(Path('examples/data/val'), fmt='cvat')
ds = CropTileDataset(recs, Path('examples/data/val/images'), train=False, output_stride=cfg.output_stride, sigma=cfg.sigma)
expected = {r['image_name']: int(r['predicted_count']) for r in csv.DictReader(open('notebooks/runs/inference/examples_val/counts.csv'))}
ok = True
for i, rec in enumerate(recs):
    prob = predict_prob(model, ds[i]['image'], device)
    pts, _ = decode_in_bounds(prob, rec.width, rec.height, tau=0.35, k=cfg.k, nms_radius=cfg.nms_radius, output_stride=cfg.output_stride)
    delta = abs(len(pts) - expected[rec.name])
    ok &= delta <= 1
    print(f'{rec.name}: cloud {len(pts):3d} | committed {expected[rec.name]:3d} | Δ {delta}')
loader = DataLoader(ds, batch_size=1, collate_fn=collate_val)
summ, _ = evaluate(model, loader, device, tau=0.35, k=cfg.k, nms_radius=cfg.nms_radius, output_stride=cfg.output_stride, match_radius_px=cfg.match_radius_px)
print({k: round(v, 3) for k, v in summ.items()})
assert ok, 'converted backbone does NOT reproduce the committed wheat counts — stop here'
print('backbone verified by metric reproduction ✓')

## 5 · CFD metadata → per-source manifest

47 MB zipped; the JSON inside is streamed (`ijson`), never `json.load`-ed. The manifest is the source for every subsetting decision in the memo (empty-image fraction, `is_train` balance, box-size percentiles, stride-4 centre-cell collision rate, licence).

In [ ]:
META = f'{CFD}/community_fish_detection_dataset.json.zip'
if not os.path.exists(META):
    !wget -q --show-progress -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
t0 = time.time()
!python -m cropcounter.cfd manifest --metadata {META} --out {CFD}/manifest
print(f'manifest in {time.time()-t0:.0f}s')
for f in os.listdir(f'{CFD}/manifest'):
    shutil.copy(f'{CFD}/manifest/{f}', f'{DRIVE}/results/manifest/{f}')

## 6 · Subset — all of Brackish, honouring the published `is_train` split

In [ ]:
import csv, json
# Resolve the exact `dataset` field string for Brackish from the manifest rather than hard-coding it.
rows = list(csv.DictReader(open(f'{CFD}/manifest/manifest.csv')))
name_col = next(c for c in rows[0].keys() if c.lower() in ('dataset', 'source', 'name'))
BRACKISH_SOURCE = next(r[name_col] for r in rows if 'rackish' in r[name_col])
print('Brackish source string:', repr(BRACKISH_SOURCE))
!rm -rf {DATA}
!python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources "{BRACKISH_SOURCE}" --train-cap 100000 --val-cap 100000 --seed 0
print(json.dumps(json.load(open(f'{DATA}/subset_summary.json')), indent=1)[:2000])
!wc -l {DATA}/download_list.txt

## 7 · Fetch images to the VM disk, resized to long side 1024 on write

Resize at fetch, not at train: (i) fairness — the released baselines run at 1024/640 and AP is acutely resolution-sensitive for small objects; (ii) train/val scale consistency; (iii) disk. Boxes and `width`/`height` are rescaled into `annotations.json`; the native-scale file is kept as `annotations.native.json`.

In [ ]:
t0 = time.time()
!python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs
print(f'fetched in {time.time()-t0:.0f}s')
!du -sh {DATA}/train/images {DATA}/val/images
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## 8 · Throughput probe — is the frozen path CPU-bound?

Measures images/s and GPU utilisation for a few `num_workers` settings on the real loader *before* any long run. If GPU utilisation sits well under ~60 % at the best worker count, do not buy a bigger GPU for the frozen runs — spend it on the unfrozen comparator (Protocol B/C).

In [ ]:
import subprocess, threading, statistics, torch, time
from cropcounter.train import TrainConfig, build_loaders, build_model, resolve_device
from cropcounter.dinov3_pyramid import autocast_context

base = TrainConfig.from_json('examples/config_box_brackish.json')
base.data_root = pathlib.Path(DATA); base.weights_dir = pathlib.Path('weights')
device = resolve_device(None)
model = build_model(base, device); model.train()

def gpu_util_sampler(stop, out):
    while not stop.is_set():
        try:
            u = subprocess.check_output(['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits']).decode().strip()
            out.append(int(u.split()[0]))
        except Exception:
            pass
        time.sleep(0.5)

results = {}
for nw in sorted({2, 4, min(8, os.cpu_count()), os.cpu_count()}):
    cfg = TrainConfig.from_dict(base.to_dict()); cfg.num_workers = nw
    train_loader, *_ = build_loaders(cfg, device)
    it = iter(train_loader)
    next(it)  # warm-up (worker spin-up)
    stop, samples = threading.Event(), []
    th = threading.Thread(target=gpu_util_sampler, args=(stop, samples)); th.start()
    n_img, t0 = 0, time.time()
    for _ in range(15):
        images, targets, _n = next(it)
        images = images.to(device, non_blocking=True)
        with autocast_context(device):
            out = model(images)
        torch.cuda.synchronize()
        n_img += images.shape[0]
    dt = time.time() - t0
    stop.set(); th.join()
    results[nw] = (n_img / dt, statistics.mean(samples) if samples else float('nan'))
    print(f'num_workers={nw:2d}: {n_img/dt:6.1f} img/s | GPU util {results[nw][1]:.0f}%')
    del it, train_loader
BEST_WORKERS = max(results, key=lambda k: results[k][0])
print('→ use num_workers =', BEST_WORKERS)
json.dump({str(k): v for k, v in results.items()}, open(f'{DRIVE}/results/throughput_probe.json', 'w'), indent=1)

## 9 · Train the frozen-trunk head on Brackish `is_train` (the go/no-go run)

Config = `examples/config_box_brackish.json` (trimmed here to 8 epochs × 1 tile/frame for a free-tier session) with `augment_profile: natural` (no vertical flips / 90° rotations underwater), log-space size parameterisation, `negative_tile_fraction 0.2` (the empty-image hazard is handled in the sampler, not the loss). Checkpoints, `history.json`, `curves.png` and `predictions.json` land on Drive as they are written.

In [ ]:
cfg = TrainConfig.from_json('examples/config_box_brackish.json')
cfg.data_root = pathlib.Path(DATA); cfg.weights_dir = pathlib.Path('weights')
cfg.out_dir = pathlib.Path(f'{DRIVE}/runs'); cfg.run_name = 'brackish_frozen_s0'
cfg.num_workers = BEST_WORKERS; cfg.seed = 0
# Free-tier Colab caps a session at ~5 h; 8 epochs x 1 tile/frame (~11.5k tiles/epoch on 960x540 frames)
# is enough for a go/no-go signal and leaves room for the linear probe + baseline in the same session.
cfg.epochs = 8; cfg.tiles_per_image = 1
cfg.to_json(pathlib.Path('/content/config_brackish_frozen.json'))
print(json.dumps(cfg.to_dict(), indent=1))

In [ ]:
!python -m cropcounter.train --config /content/config_brackish_frozen.json 2>&1 | tee {DRIVE}/runs/brackish_frozen_s0.log

## 10 · Linear-probe diagnostic — freeze the fuse trunk too (2 epochs)

Non-trivial AP here ⇒ the DINOv3 features are *near-linearly* box-decodable and the thesis is strong. Near-zero while the full decoder works ⇒ the story is about the decoder, not the features — a materially different memo.

In [ ]:
cfg_lp = TrainConfig.from_dict(cfg.to_dict())
cfg_lp.freeze_fusion = True; cfg_lp.epochs = 2; cfg_lp.warmup_epochs = 0; cfg_lp.run_name = 'brackish_linearprobe_s0'
cfg_lp.to_json(pathlib.Path('/content/config_brackish_linearprobe.json'))
!python -m cropcounter.train --config /content/config_brackish_linearprobe.json 2>&1 | tee {DRIVE}/runs/brackish_linearprobe_s0.log

## 11 · Baseline — released RF-DETR-Nano (640) through the *identical* harness on the *identical* val images

Weights from the community-fish-detector GitHub release (Apache inference licence). Scored with our `det_metrics.coco_eval` against the same `val/annotations.json` — the same function scores ours. Threshold 0.001 so the PR curve is complete. Optionally repeat for RF-DETR-Small (1024).

In [ ]:
!pip install -q rfdetr supervision
import urllib.request
BASELINES = {
  'rfdetr_nano_640':  'https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-nano-640-2026.02.02.cp-011.20260706-release.pth',
  # 'rfdetr_small_1024': 'https://github.com/filippovarini/community-fish-detector/releases/download/2026.07.06-release/cfd-rf-detr-small-1024-2026.06.06.cp-016.20260706-release.pth',
}
os.makedirs(f'{DRIVE}/weights/baselines', exist_ok=True)
for name, url in BASELINES.items():
    p = f'{DRIVE}/weights/baselines/{name}.pth'
    if not os.path.exists(p):
        urllib.request.urlretrieve(url, p)
    print(name, os.path.getsize(p) / 1e6, 'MB')

In [ ]:
from rfdetr import from_checkpoint
from PIL import Image
from tqdm.auto import tqdm
from cropcounter.det_metrics import coco_eval, write_coco_results

gt = json.load(open(f'{DATA}/val/annotations.json'))
id_by_name = {os.path.basename(im['file_name']): im['id'] for im in gt['images']}
baseline_metrics = {}
for name in BASELINES:
    det_model = from_checkpoint(f'{DRIVE}/weights/baselines/{name}.pth')
    dets, t0 = [], time.time()
    for fname, image_id in tqdm(id_by_name.items(), desc=name):
        img = Image.open(f'{DATA}/val/images/{fname}').convert('RGB')
        d = det_model.predict(img, threshold=0.001)
        for (x1, y1, x2, y2), s in zip(d.xyxy, d.confidence):
            dets.append({'image_id': image_id, 'category_id': 1,
                         'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s)})
    secs = time.time() - t0
    write_coco_results(dets, f'{DRIVE}/results/{name}_predictions.json')
    m = coco_eval(f'{DATA}/val/annotations.json', dets)
    m['seconds_per_image'] = secs / len(id_by_name)
    baseline_metrics[name] = m
    print(name, {k: round(v, 4) for k, v in m.items()})
json.dump(baseline_metrics, open(f'{DRIVE}/results/baseline_metrics.json', 'w'), indent=1)

## 12 · Results table — ours vs baseline, same images, same scorer

In [ ]:
import csv
from cropcounter.det_metrics import coco_eval, read_coco_results
from cropcounter.dinov3_pyramid import PyramidDecoder
rows = []
for run in ('brackish_frozen_s0', 'brackish_linearprobe_s0'):
    rd = f'{DRIVE}/runs/{run}'
    if not os.path.exists(f'{rd}/history.json'):
        continue
    h = json.load(open(f'{rd}/history.json'))
    best = min(range(len(h['val_loss'])), key=lambda i: h['val_loss'][i])
    row = {'model': run, 'input': '1024 long side', 'epoch': best + 1}
    for k in ('val_ap', 'val_ap50', 'val_ap75', 'val_ar100', 'val_f1', 'val_precision', 'val_recall', 'val_count_mae'):
        row[k.replace('val_', '')] = round(h[k][best], 4) if k in h else None
    # Re-score the saved predictions with the same function used for the baseline — the parity check.
    if os.path.exists(f'{rd}/predictions.json'):
        row['ap_rescored'] = round(coco_eval(f'{DATA}/val/annotations.json', read_coco_results(f'{rd}/predictions.json'))['ap'], 4)
    rows.append(row)
for name, m in baseline_metrics.items():
    rows.append({'model': f'{name} (released, likely saw these images)', 'input': name.split('_')[-1],
                 **{k: round(m[k], 4) for k in ('ap', 'ap50', 'ap75', 'ar100')}})
# Parameter accounting — trainable AND total (the frozen ~89M backbone runs on every forward).
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_dec_box = sum(p.numel() for p in PyramidDecoder((128, 256, 512, 1024), task='box').parameters())
for r in rows:
    if r['model'].startswith('brackish'):
        r['trainable_params_M'] = round(n_dec_box / 1e6, 2); r['total_params_M'] = round((n_backbone + n_dec_box) / 1e6, 1)
keys = sorted({k for r in rows for k in r}, key=lambda k: (k != 'model', k))
with open(f'{DRIVE}/results/results.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
print('| ' + ' | '.join(keys) + ' |'); print('|' + '---|' * len(keys))
for r in rows:
    print('| ' + ' | '.join(str(r.get(k, '')) for k in keys) + ' |')

## 13 · Inference FLOPs (measured, not quoted) — ours at 1024×576

In [ ]:
from torch.utils.flop_counter import FlopCounterMode
box_model = build_model(cfg, device); box_model.eval()
x = torch.randn(1, 3, 576, 1024, device=device)
with torch.no_grad(), FlopCounterMode(display=False) as fc:
    box_model(x)
print(f'ours (frozen ConvNeXt-B + decoder, 1024x576): {fc.get_total_flops() / 1e9:.1f} GFLOPs')
try:
    core = det_model.model.model  # rfdetr wraps a LWDETR nn.Module
    xr = torch.randn(1, 3, 640, 640, device=device)
    with torch.no_grad(), FlopCounterMode(display=False) as fc2:
        core(xr)
    print(f'RF-DETR-Nano (640x640): {fc2.get_total_flops() / 1e9:.1f} GFLOPs')
except Exception as e:
    print('RF-DETR FLOP count skipped:', repr(e)[:200])

## 14 · Visual spot checks — GT (green) · ours (red) · RF-DETR (blue), six val images spanning the count range

In [ ]:
import numpy as np, matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from collections import defaultdict
ours = defaultdict(list); theirs = defaultdict(list); gts = defaultdict(list)
for d in read_coco_results(f'{DRIVE}/runs/brackish_frozen_s0/predictions.json'):
    if d['score'] >= 0.3: ours[d['image_id']].append(d['bbox'])
for d in read_coco_results(f'{DRIVE}/results/rfdetr_nano_640_predictions.json'):
    if d['score'] >= 0.3: theirs[d['image_id']].append(d['bbox'])
for a in gt['annotations']:
    gts[a['image_id']].append(a['bbox'])
name_by_id = {im['id']: os.path.basename(im['file_name']) for im in gt['images']}
by_count = sorted(name_by_id, key=lambda i: len(gts[i]))
picks = [by_count[int(q * (len(by_count) - 1))] for q in (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)]
fig = Figure(figsize=(18, 12)); FigureCanvasAgg(fig)
for ax, iid in zip(fig.subplots(2, 3).ravel(), picks):
    ax.imshow(Image.open(f"{DATA}/val/images/{name_by_id[iid]}"))
    for boxes, col in ((gts[iid], 'lime'), (ours[iid], 'red'), (theirs[iid], 'deepskyblue')):
        for x, y, w, h in boxes:
            ax.add_patch(matplotlib.patches.Rectangle((x, y), w, h, fill=False, edgecolor=col, linewidth=1.2))
    ax.set_title(f"{name_by_id[iid]}  gt {len(gts[iid])} · ours {len(ours[iid])} · rfdetr {len(theirs[iid])}", fontsize=9); ax.axis('off')
fig.savefig(f'{DRIVE}/results/viz/spot_checks.png', dpi=110, bbox_inches='tight')
from IPython.display import Image as IPImage, display
display(IPImage(f'{DRIVE}/results/viz/spot_checks.png'))

## 15 · Hand-off

Everything the memo needs is now on Drive: `results/results.csv`, `results/baseline_metrics.json`, `results/manifest/`, `results/throughput_probe.json`, `results/viz/spot_checks.png`, and the two run dirs. Paste the table from cell 12 into `output/frozen-trunk-detection/report.md` § Results and apply the kill criterion: frozen head AP50 < ~0.5 while RF-DETR-Nano > ~0.8 ⇒ the frozen-trunk premise fails underwater → rescope to partial unfreeze or the label-efficiency claim alone.